# Notebook 06 — prospective evaluation protocol and faculty review integration

This notebook prepares **blinded, version-locked review packets** from the authored Notebook 04 and 05 fixtures and defines the import path for later consenting learner encounters. It contains no independent faculty ratings, no observed learner outcomes and no trained student-action model. The protocol in [`study_protocol_v1.json`](../app/data/study_protocol_v1.json) is **proposed**, pending faculty approval and an institutional ethics determination. Revise it as a new protocol version before collecting outcomes.

**Sources:** [`AGENTS.md`](../../AGENTS.md), [`context.md`](../../context.md) §14 and §15, [`PRD.md`](../../PRD.md) §§9 and 15 (V-01–V-04), [Notebook 01](../docs/validation/dataset_audit.json), [02](../docs/validation/action_data.json), [03](../docs/validation/patient_environment.json), [04](../docs/validation/grounding_eval.json), [05](../docs/validation/debrief_eval.json). The three authored [`cases/`](../../cases/) and shared `backend/` case, encounter, feedback and rubric contracts are hashed into the manifest. The unlicensed Patient–Doctor contents are excluded.

In [ ]:
import os, sys, json
from pathlib import Path
os.environ["LLM_ENABLED"] = "false"
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "PRD.md").exists())
sys.path.insert(0,str(ROOT)); sys.path.insert(0,str(ROOT/"backend"))
from backend_kelsch.scripts.eval_study import evaluate
from backend_kelsch.app.study.protocol import PROTOCOL, lock_manifest, make_packets, split_audit
from backend_kelsch.app.study.analysis import planning_n_per_arm, wilson
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt
report, packets, audit_events = evaluate()
print("Protocol", report["protocol"]["protocol_id"], report["protocol"]["protocol_status"])
print("Lock",report["protocol"]["lock_id"], "Source files",len(report["protocol"]["source_sha256"]))
print("Review rows", report["reviews"]["submitted_rows"], "Audit events",len(audit_events))
assert report["reviews"]["submitted_rows"] == 0

## 1. Source lock and reviewer blindness

The lock hashes the proposed protocol, case/rubric/vocabulary files, shared and notebook code contracts, and Notebook 01–05 reports and trace ledgers. Changing any of them changes packet IDs and invalidates stale reviews. The reviewer sees the case reference, transcript context and question. Release packets omit actual releases and level labels; semantic packets omit verifier verdicts; domain packets omit provisional scores. An opaque packet ID maps back to the source only in the analysis process.

**Sequential release judgment:** review the whole case and preceding student utterances from the beginning. An individual turn has no visible engine disclosure answer. Reviewers should derive the expected state independently from the approved case and protocol, then compare decisions after both reviews are sealed.

In [ ]:
display(pd.DataFrame([{"kind":k,"pending":v,"resolved":report["reviews"]["resolved_by_kind"].get(k,0)} for k,v in report["packet_counts"].items()]))
for kind in ("release","semantic","claim","domain"):
    p = next(p for p in packets if p["kind"]==kind)
    print(kind, "packet",p["packet_id"], "case",p["case_id"],"source",p["case_sha256"][:12], "answer",p["expected"])
fig, ax=plt.subplots(figsize=(8,3.5))
labels=list(report["packet_counts"]); vals=[report["packet_counts"][k] for k in labels]
ax.bar(labels,vals,color="#377f99"); ax.set(ylabel="Pending packets",title="Source-locked review workload (no ratings imported)")
for i,v in enumerate(vals): ax.text(i,v+2,str(v),ha="center")
fig.tight_layout(); plt.show()

## 2. Independent review and adjudication

Each packet requires two independent pseudonymous faculty IDs, timestamps with time zones, exact protocol/case hashes, and a typed answer. Agreement resolves a packet. Disagreement waits for a distinct adjudicator; unadjudicated or partial reviews produce denominators but no aggregate gate. Clinical release labels are lists of approved fact IDs, semantic and debrief-claim labels are booleans, action labels are exact student spans in the `student_skill` namespace, and domain ratings are 1–4 with verbatim evidence or `N/A` with a reason. General dialogue acts remain a separate namespace. The audit chain stores only hashes of answers and pseudonymous reviewers; the organizational study coordinator must independently verify identity and blinding.

In [ ]:
from collections import Counter
from backend_kelsch.app.study.protocol import load_previous
from backend_kelsch.app.study.review import load_reviews
p=next(x for x in packets if x["kind"]=="release")
print("Example blinded release packet:", {k:v for k,v in p.items() if k!="context"})
print("Student context:",p["context"]["student_utterance"])
print("No review imports ->", report["reviews"])
print("No estimates ->", {k:v for k,v in report["results"].items() if v is not None})
assert all(v=="NOT EVALUABLE" for k,v in report["gates"].items() if k!="protocol_and_import_workflow")

## 3. Prospective learner import, consent and leakage control

The optional private corpus must specify documented opt-in and institutional ethics determination, synthetic-only cases, deidentification, consent reference per conversation, pseudonymous learner assignment, source digest, and the locked case hash/version. Do not commit raw transcripts, audio, consent linkage or reviewer identities. Redact incidental identifying speech before annotation. Refusal must not affect grading. The coordinator keeps original material and retention policy in institution-controlled storage.

The shared [`grouped_split`](../app/action_data/preparation.py) unions patient ID, script family and normalized duplicate turn text before a seeded train/validation/test split. Notebook 06 rejects missing identifiers and cross-split links. The three current authored patients give at most three independent patient groups, so they cannot support a credible trained extraction evaluation. New approved cases and a new protocol lock are needed. Only fully adjudicated human action spans can reach the baseline; prompted predictions must be independently logged with model ID and prompt version on the exact held-out annotation IDs.

In [ ]:
spec=json.loads(PROTOCOL.read_text())
print("Population:",spec["population"])
print("Privacy controls:")
for key,value in spec["privacy"].items(): print("-",key+":",value)
print("Prospective conversations imported:",report["prospective_conversations"])
print("Model metrics:",report["results"]["action"])
print("Whole-patient split:",split_audit([]))

## 4. Preregistered outcomes and sample-size rationale

**Future primary outcome:** between-arm difference in each learner's mean baseline-to-held-out-case change per common faculty-scored rubric domain. Randomized parallel assignment to feedback versus standard practice; blinded faculty scores; learner is the unit. Missing visits/scores are counted by arm, never set to zero. A descriptive normal 95% interval is available once each arm has at least two reviewed changes, but an efficacy gate needs an approved protocol and adequate randomized sample. Secondary outcomes cover held-out action recall, disclosure release precision/recall, semantic verdicts, faculty scoring agreement and faculty-approved difficulty presets.

The PRD's 15–20 hand-scored transcripts are a *feasibility* target, not powered efficacy. This proposal chooses 20 distinct double-rated encounters per domain as a floor for estimating quadratic weighted Cohen kappa with an encounter-cluster bootstrap interval. A hypothetical standardized effect of 0.5, two-sided 5% alpha and 80% power suggests roughly 63 learners per arm, or 74 allowing 15% loss. This assumption must be reviewed before recruitment. About 100 independent binary judgments would give a worst-case normal 95% half-width near 0.10; repeated facts in turns are correlated, so Wilson intervals are descriptive until enough case clusters exist.

In [ ]:
display(pd.DataFrame([{"quantity":"grader feasibility","planned":"20 distinct encounters/domain","observed":0},
    {"quantity":"future efficacy per arm","planned":planning_n_per_arm()["with_loss_rounded_up"],"observed":0},
    {"quantity":"release review turns","planned":report["packet_counts"]["release"],"observed":0},
    {"quantity":"semantic candidates","planned":report["packet_counts"]["semantic"],"observed":0}]))
print("Wilson for 0/0:",wilson(0,0),"; no confidence interval without reviewed observations")
print("Future planning calculation:",planning_n_per_arm())

## 5. Decision trace and progression

Notebook 05's authored branches can be re-examined for reproducibility, but they are not a difficulty study or learning outcome. Full faculty review of all 189 release turns is required before clinical disclosure accuracy. All 15 semantic candidates need faculty labels and every mechanically permitted reply needs a logged judge verdict. All 41 debrief claims need independent resolution. Six authored debrief scenarios cannot meet 20 distinct faculty-scored transcripts per domain. Student action training needs a reviewed, leakage-free train/test split with every label represented; no profiles or scripted turns may substitute for observed learner actions.

In [ ]:
grounding,debrief=load_previous()
display(pd.DataFrame([{"case":r["case_id"],"scenario":r["scenario"],
    "taken":r["branch"]["taken_id"],"uncertain":",".join(r["branch"]["uncertain_predicates"]),
    "replay":r["branch"]["replay_match"],"case_sha":r["case_source"]["sha256"][:12]} for r in debrief]))
print("Current gates:",report["gates"])
assert len(debrief)==6 and len(grounding["runs"])==27

## How to run and import

From the repository root, run `LLM_ENABLED=false PYTHONPATH=backend python -m backend_kelsch.scripts.eval_study --export-packets /tmp/notebook06_pending.jsonl`. Only authorized faculty should see an export containing a prospective transcript. Do not fill labels from engine outputs. With a coordinator-approved, private corpus, use `--corpus private_corpus.json`, then independently obtain two review rows per packet and adjudicate disagreements using `--reviews private_reviews.jsonl --audit-log /tmp/notebook06_audit.jsonl`. The review importer checks source/protocol locks and exact evidence; it cannot authenticate real-world reviewer identity. Compare `--llm-predictions` and `--judge-predictions` only after complete reviews. The report writes aggregates to [`study_protocol.json`](../docs/validation/study_protocol.json); see the [readable report](../docs/validation/study_protocol.md) and [Notebook 07 prompt](../docs/notebook_07_prompt.txt).

**Gate:** PASS for the packet/source/import mechanics if all invariants and tests hold; FAIL on any broken hash, evidence span, leak or replay. Faculty feedback and scoring, trained extraction, clinical disclosure, semantic grounding, difficulty and learning remain **NOT EVALUABLE** without their independent reviews and approved cohort. Seven HTTP API tests collected successfully, but the local run timed out after 25 seconds with no completed tests. They are not counted as passing.